# Trust Intelligence：在 Colab 跑 Qwen3.8-27B

用 VS Code 的 Colab 擴充連到 **L4 或 A100** 執行階段，由上往下執行。

最後一格會印出一個 `https://xxxx.trycloudflare.com` 網址，把它加上 `/v1` 貼到專案 `.env` 的 `LLM_BASE_URL`，`LLM_MODEL` 填下面的 `MODEL`，然後重新啟動後端。

> Colab 斷線或重開後網址會改變，要重新執行並更新 `.env`。後端連不上模型時會自動改用規則式回答，Demo 不會中斷。

In [ ]:
# 1. 確認 GPU（27B 量化版約需 17 GB 以上 VRAM）
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 安裝 Ollama 與 cloudflared
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# 3. 啟動 Ollama 伺服器（背景執行）
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '8192'   # Agent 需要較長的上下文
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'         # 模型常駐，避免每次重新載入
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s http://localhost:11434/api/version

In [ ]:
# 4. 下載模型（第一次約需數分鐘）
# ⚠ 請把 MODEL 改成你實際使用的 Qwen3.8-27B 模型名稱（Ollama 的 tag，或 hf.co/<repo>:<量化> 的寫法）
MODEL = 'qwen3.8:27b'
!ollama pull {MODEL}

In [ ]:
# 5. 測試：中文回答＋工具呼叫
import json, urllib.request
body = {
  'model': MODEL,
  'messages': [{'role': 'user', 'content': '前輪胎為什麼要換？請先查資料'}],
  'tools': [{'type': 'function', 'function': {'name': 'get_item_detail', 'description': '查檢查項目',
            'parameters': {'type': 'object', 'properties': {'part_name': {'type': 'string'}}, 'required': ['part_name']}}}],
}
req = urllib.request.Request('http://localhost:11434/v1/chat/completions', json.dumps(body).encode(), {'Content-Type': 'application/json'})
msg = json.load(urllib.request.urlopen(req))['choices'][0]['message']
print(json.dumps(msg, ensure_ascii=False, indent=2))
# 看到 tool_calls 裡有 get_item_detail 就代表工具呼叫正常

In [ ]:
# 6. 開通道：把印出的網址加上 /v1 貼到 .env 的 LLM_BASE_URL
import re
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n貼到 .env：')
        print(f'LLM_BASE_URL={m.group(0)}/v1')
        print(f'LLM_MODEL={MODEL}')
        break

In [ ]:
# 7. 保持執行（這格跑著，通道就不會關）
while True:
    time.sleep(60)